## Matrix Multiplication — The Boss Fight
This is where Triton shines. MatMul is THE kernel for LLM inference, and understanding the tiled approach is essential.

### Tiled MatMul Strategy
For `C = A @ B` where `A` is `(M, K)` and `B` is `(K, N)`:
```python
For each output tile C[m : m + BLOCK_M, n : n + BLOCK_N]:
    accumulator = zeros(BLOCK_M, BLOCK_N)

    for k in range(0, K, BLOCK_K):
        a_tile = A[m : m + BLOCK_M, k : k + BLOCK_K]
        b_tile = B[k : k + BLOCK_K, n : n + BLOCK_N]

        accumulator += a_tile @ b_tile

    C[m : m + BLOCK_M, n : n + BLOCK_N] = accumulator

```
- `BLOCK_M` = tile height along `M` dimension
- `BLOCK_N` = tile width along `N` dimension
- `BLOCK_K` = reduction tile size along `K` dimension

The key insight: each output tile only needs `BLOCK_M*K + K*BLOCK_N` loads from HBM but does `BLOCK_M*BLOCK_N*K` FLOPs. By making tiles large enough, you become compute-bound instead of memory-bound.

## The kernel

In [1]:
import triton
import triton.language as tl

@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M,N,K,
    stride_am, stride_ak,
    stride_bk, stride_bn, 
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
   pid_m = tl.program_id(axis=0)
   pid_n = tl.program_id(axis=1)

   offsets_m = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
   offsets_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
   offsets_k = tl.arange(0, BLOCK_SIZE_K)

   #calculate the ptr of each element for each tiled matrix
   a_ptrs = a_ptr + (offsets_m[:, None]*stride_am + offsets_k[None, :]*stride_ak)
   b_ptrs = b_ptr + (offsets_k[:, None]*stride_bk + offsets_n[None, :]*stride_bn)
   c_ptrs = c_ptr + (offsets_m[:, None]*stride_cm + offsets_n[None, :]*stride_cn)

   accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

   for k in range(0, K, BLOCK_SIZE_K):
      a_mask = (offsets_m[:, None] < M) & (offsets_k[None, :] + k < K)
      b_mask = (offsets_k[:, None] + k < K) & (offsets_n[None, :]< N)

      a = tl.load(a_ptrs+k*stride_ak, mask=a_mask, other=0.0)
      b = tl.load(b_ptrs+k*stride_bk, mask=b_mask, other=0.0)

      accumulator += tl.dot(a, b)

   c_mask = (offsets_m[:, None] < M) & (offsets_n[None, :] < N)
   tl.store(c_ptrs, accumulator, mask=c_mask)
    

### Concepts Introduced
- 2D Grid — `program_id(0)` and `program_id(1)` give you the M and N tile indices. The grid is `(ceil(M/BM), ceil(N/BN))`.
- 2D Pointer Arithmetic — `offsets_m[:, None] * stride_am + offsets_k[None, :] * stride_ak` creates a 2D block of pointers using broadcasting. This is the 2D equivalent of the 1D ptr + offsets pattern.
- `tl.dot(a, b)` — Triton’s block-level matrix multiply. The compiler maps this to Tensor Core mma instructions on supported hardware. This is where the actual throughput comes from.

### Launcher

In [2]:
import torch
def matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.is_cuda and b.is_cuda
    M, K = a.shape
    K, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float32)

    grid = lambda META: (
        triton.cdiv(M, META['BLOCK_SIZE_M']),
        triton.cdiv(N, META['BLOCK_SIZE_N']),
    )

    matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_SIZE_M=128,
        BLOCK_SIZE_N=128,
        BLOCK_SIZE_K=32,
    )
    return c